In [1]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("/media/baai/7E357B68494ABB7B/model/map-anything-apache-v1").to(device)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/baai/.cache/torch/hub/facebookresearch_dinov2_main


Loading weights from local directory


In [2]:
from mapanything.utils.image import load_images
# Load and preprocess images from a folder or list of paths
images = "/media/baai/7E357B68494ABB7B/model/map-anything/mapanything/test_images"  # or ["path/to/img1.jpg", "path/to/img2.jpg", ...]
views = load_images(images)
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=True,  # Trades off speed for more views (up to 2000 views on 140 GB). Trade off is negligible - see profiling section
    minibatch_size=None,              # Minibatch size for memory-efficient inference (use 1 for smallest GPU memory consumption). Default is dynamic computation based on available GPU memory.
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    use_multiview_confidence=False,   # Enable multi-view depth consistency based confidence in place of learning-based one
)

In [3]:
for i, pred in enumerate(predictions):
    # Geometry outputs
    pts3d = pred["pts3d"]                     # 3D points in world coordinates (B, H, W, 3)
    pts3d_cam = pred["pts3d_cam"]             # 3D points in camera coordinates (B, H, W, 3)
    depth_z = pred["depth_z"]                 # Z-depth in camera frame (B, H, W, 1)
    depth_along_ray = pred["depth_along_ray"] # Depth along ray in camera frame (B, H, W, 1)

    # Camera outputs
    ray_directions = pred["ray_directions"]   # Ray directions in camera frame (B, H, W, 3)
    intrinsics = pred["intrinsics"]           # Recovered pinhole camera intrinsics (B, 3, 3)
    camera_poses = pred["camera_poses"]       # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world poses in world frame (B, 4, 4)
    cam_trans = pred["cam_trans"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world translation in world frame (B, 3)
    cam_quats = pred["cam_quats"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world quaternion in world frame (B, 4)

    # Quality and masking
    confidence = pred["conf"]                 # Per-pixel confidence scores (B, H, W)
    mask = pred["mask"]                       # Combined validity mask (B, H, W, 1)
    non_ambiguous_mask = pred["non_ambiguous_mask"]                # Non-ambiguous regions (B, H, W)
    non_ambiguous_mask_logits = pred["non_ambiguous_mask_logits"]  # Mask logits (B, H, W)

    # Scaling
    metric_scaling_factor = pred["metric_scaling_factor"]  # Applied metric scaling (B,)

    # Original input
    img_no_norm = pred["img_no_norm"]    
    
    pass
 

In [5]:
import torch
import numpy as np
import open3d as o3d
import os

def to_numpy(data):
    if isinstance(data, torch.Tensor):
        return data.detach().cpu().numpy()
    return data

def save_predictions_to_ply(predictions, output_path="scene_reconstruction.ply", conf_threshold=0.5, voxel_size=0.02):
    """
    将 MapAnything 的预测结果拼接并保存为 PLY 文件 (适用于服务器环境)
    
    :param predictions: 预测结果列表
    :param output_path: 保存的文件路径 (.ply)
    :param conf_threshold: 置信度阈值 (根据模型输出范围调整，通常 0-1 或 logit)
    :param voxel_size: 体素下采样尺寸 (米)，建议 0.01~0.05
    """
    
    all_points = []
    all_colors = []
    
    print(f"[开始处理] 共有 {len(predictions)} 帧数据...")

    for i, pred in enumerate(predictions):
        # 1. 提取数据并转为 numpy
        # pts3d: (B, H, W, 3) -> 取 (H, W, 3)
        pts3d = to_numpy(pred["pts3d"])[0]
        img = to_numpy(pred["img_no_norm"])[0] 
        mask = to_numpy(pred["mask"])[0].squeeze()
        conf = to_numpy(pred["conf"])[0]
        
        # 2. 筛选有效点
        # 逻辑: Mask 必须有效，且置信度大于阈值
        # 注意：如果 mask 是 logits，可能需要 sigmoid，但通常 MapAnything 输出的是 0/1 或 float mask
        valid_mask = (mask > 0.5) & (conf > conf_threshold)
        
        # 简单的距离过滤 (可选)：去掉距离相机太远的点(比如 > 50米)，防止天空球造成的伪影
        # 这里仅作示例，如果不需要可注释掉
        # depth = to_numpy(pred["depth_z"])[0].squeeze()
        # valid_mask = valid_mask & (depth < 50.0)

        if valid_mask.sum() == 0:
            continue

        # 3. 提取点和颜色
        pts_flat = pts3d[valid_mask]
        
        # 处理颜色: 确保是 0-1 范围的 float (Open3D 要求)
        if img.max() > 1.0:
            colors_flat = img[valid_mask] / 255.0
        else:
            colors_flat = img[valid_mask]

        all_points.append(pts_flat)
        all_colors.append(colors_flat)
        
        if (i + 1) % 10 == 0:
            print(f"  - 已处理 {i + 1} 帧")

    # --- 合并与优化 ---
    if not all_points:
        print("[错误] 没有提取到有效的点云数据，未保存文件。")
        return

    print("[正在合并] 连接所有帧的点云...")
    all_points = np.concatenate(all_points, axis=0)
    all_colors = np.concatenate(all_colors, axis=0)
    
    print(f"  - 原始点数: {all_points.shape[0]}")

    # 创建 Open3D 点云对象
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(all_points)
    pcd.colors = o3d.utility.Vector3dVector(all_colors)

    # 1. 体素下采样 (关键步骤：减少冗余点，统一密度)
    print(f"[正在优化] 体素下采样 (Voxel Size: {voxel_size}m)...")
    pcd = pcd.voxel_down_sample(voxel_size=voxel_size)
    print(f"  - 下采样后点数: {len(pcd.points)}")

    # 2. 离群点移除 (去除空中的噪点)
    print("[正在优化] 统计式离群点移除...")
    # nb_neighbors: 考虑周围多少个点, std_ratio: 标准差倍数，越小过滤越严格
    pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

    # --- 保存文件 ---
    print(f"[保存] 正在写入文件: {output_path}")
    # write_ascii=False 使用二进制保存，文件更小，读写更快
    o3d.io.write_point_cloud(output_path, pcd, write_ascii=False)
    print(f"✅ 成功! 文件已保存，请下载 {output_path} 使用 MeshLab 查看。")

# --- 使用示例 ---
save_predictions_to_ply(predictions, "output_scene.ply", conf_threshold=0.5, voxel_size=0.02)

[开始处理] 共有 6 帧数据...
[正在合并] 连接所有帧的点云...
  - 原始点数: 1151492
[正在优化] 体素下采样 (Voxel Size: 0.02m)...
  - 下采样后点数: 537800
[正在优化] 统计式离群点移除...
[保存] 正在写入文件: output_scene.ply
✅ 成功! 文件已保存，请下载 output_scene.ply 使用 MeshLab 查看。


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d

def to_numpy(data):
    """
    辅助函数：将 Tensor 转换为 Numpy 数组，并处理设备和维度
    """
    if isinstance(data, torch.Tensor):
        return data.detach().cpu().numpy()
    return data

def visualize_mapanything_output(predictions, visualize_3d=True):
    """
    可视化 MapAnything 的预测结果
    :param predictions: 预测结果列表
    :param visualize_3d: 是否开启 Open3D 点云可视化 (需要安装 open3d)
    """
    
    print(f"Total predictions to process: {len(predictions)}")

    for i, pred in enumerate(predictions):
        print(f"--- Processing Frame {i} ---")
        
        # --- 1. 数据提取与预处理 (去除 Batch 维度并转为 Numpy) ---
        # 假设 Batch size B=1，我们需要取 [0]
        
        # 图像 (H, W, 3)
        img = to_numpy(pred["img_no_norm"])[0] 
        # 如果图像是 float (0-1)，转换为 uint8 (0-255) 以便显示
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
            
        # 深度 (H, W) - 去掉最后的一维通道
        depth = to_numpy(pred["depth_z"])[0].squeeze()
        
        # 置信度 (H, W)
        conf = to_numpy(pred["conf"])[0]
        
        # Mask (H, W)
        mask = to_numpy(pred["mask"])[0].squeeze()
        
        # 3D 点云 (H, W, 3)
        pts3d = to_numpy(pred["pts3d"])[0]
        
        # 相机姿态 (4, 4)
        pose = to_numpy(pred["camera_poses"])[0]

        # --- 2. Matplotlib 2D 面板可视化 ---
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f'Prediction Visualization - Frame {i}', fontsize=16)

        # A. 原始图像
        axes[0, 0].imshow(img)
        axes[0, 0].set_title("Input Image (RGB)")
        axes[0, 0].axis('off')

        # B. 深度图 (Z-Depth)
        # 使用 percentile 过滤极值，使可视化对比度更好
        vmin, vmax = np.percentile(depth[mask > 0], [2, 98]) if mask.sum() > 0 else (depth.min(), depth.max())
        im_depth = axes[0, 1].imshow(depth, cmap='magma', vmin=vmin, vmax=vmax)
        axes[0, 1].set_title("Depth Z (Camera Frame)")
        axes[0, 1].axis('off')
        plt.colorbar(im_depth, ax=axes[0, 1], fraction=0.046, pad=0.04)

        # C. 置信度
        im_conf = axes[0, 2].imshow(conf, cmap='viridis', vmin=0, vmax=1)
        axes[0, 2].set_title("Confidence Map")
        axes[0, 2].axis('off')
        plt.colorbar(im_conf, ax=axes[0, 2], fraction=0.046, pad=0.04)

        # D. 掩码 (Mask)
        axes[1, 0].imshow(mask, cmap='gray')
        axes[1, 0].set_title("Validity Mask")
        axes[1, 0].axis('off')

        # E. 非模糊区域 (Non-Ambiguous)
        non_amb_mask = to_numpy(pred["non_ambiguous_mask"])[0]
        axes[1, 1].imshow(non_amb_mask, cmap='gray')
        axes[1, 1].set_title("Non-Ambiguous Mask")
        axes[1, 1].axis('off')

        # F. 3D 点云投影 (简单的 X, Y 散点图，或者空置)
        # 这里我们简单展示 Mask 后的深度直方图，了解分布
        valid_depth = depth[mask > 0.5]
        if len(valid_depth) > 0:
            axes[1, 2].hist(valid_depth.flatten(), bins=50, color='skyblue', edgecolor='black')
            axes[1, 2].set_title("Depth Distribution (Valid Pixels)")
        else:
            axes[1, 2].text(0.5, 0.5, "No Valid Depth", ha='center')
        
        plt.tight_layout()
        plt.show()

        # --- 3. Open3D 3D 点云可视化 (可选) ---
        if visualize_3d:
            try:
                print("Generating 3D Point Cloud...")
                
                # 展平数组
                H, W, _ = pts3d.shape
                xyz = pts3d.reshape(-1, 3)
                colors = img.reshape(-1, 3) / 255.0  # Open3D 需要 0-1 的 float 颜色
                valid_mask = mask.reshape(-1) > 0.5  # 只显示 mask 为有效的部分

                # 过滤无效点
                xyz = xyz[valid_mask]
                colors = colors[valid_mask]

                # 创建点云对象
                pcd = o3d.geometry.PointCloud()
                pcd.points = o3d.utility.Vector3dVector(xyz)
                pcd.colors = o3d.utility.Vector3dVector(colors)

                # 添加坐标轴 (原点)
                axes_mesh = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])

                # 创建可视化窗口
                # 注意：这会弹出一个窗口，你需要手动关闭它程序才会继续
                o3d.visualization.draw_geometries(
                    [pcd, axes_mesh], 
                    window_name=f"Frame {i} - 3D Point Cloud",
                    width=1024, height=768,
                    left=50, top=50
                )
                
            except Exception as e:
                print(f"Could not visualize 3D: {e}")
                print("Make sure open3d is installed correctly.")

# --- 使用示例 ---
# 假设 'predictions' 已经在你的上下文中定义好了
# visualize_mapanything_output(predictions)